Step 0. 라이브러리 설치 및 설정

In [8]:
!pip install -U langchain-community langchain-chroma


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import pandas as pd
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# 수정된 경로
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
# OpenAI API Key 설정
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

Step 1 & 2. 속성 추출(ABSA) 및 감성 분석

In [10]:
# 전처리된 데이터 로드
df = pd.read_csv('./final_reviewdata.csv')

# 예시: LLM을 활용한 속성 분석 함수 (실제 적용 시에는 비용 고려하여 주요 샘플에 적용)
def extract_aspect_sentiment(text):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = f"""
    아래 게임 리뷰에서 다음 속성들에 대해 긍정/부정/중립을 판단하고 스코어(1~5)를 매겨줘.
    속성: [스토리, 그래픽, 최적화, 밸런스, 타격감]
    리뷰: {text}
    출력 형식: JSON 형식 (예: {{"스토리": 5, "그래픽": 3, ...}})
    """
    # 실제 구현 시에는 이 함수를 apply로 실행하되, 비용 절감을 위해 사전 정의된 키워드 기반 스코어링 추천
    pass

print("속성 분석 준비 완료")


속성 분석 준비 완료


Step 3. Vector Embedding 및 Vector DB 저장

In [11]:
# 3-1. LangChain Document 객체 생성
documents = []
for i, row in df.sample(2000).iterrows(): # 우선 2,000건 샘플로 테스트 추천
    meta = {
        "game_name": row['게임명'],
        "genre": row['장르'],
        "recommend": row['추천여부']
    }
    doc = Document(page_content=str(row['리뷰 원문']), metadata=meta)
    documents.append(doc)

# 3-2. Vector DB (Chroma) 구축 및 임베딩 저장
embeddings = OpenAIEmbeddings()
vector_db = Chroma.from_documents(
    documents, 
    embeddings, 
    persist_directory="./chroma_db"
)

print(f"Vector DB 생성 완료 (데이터: {len(documents)}건)")

Vector DB 생성 완료 (데이터: 2000건)


Step 4. RAG 쿼리 및 사용자 맞춤형 추천 생성

In [13]:
# 1. 프롬프트 템플릿 수정 (게임명 출력 강조)
retriever = vector_db.as_retriever(search_kwargs={"k": 10})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

template = """
당신은 게임 전문 추천 AI 가이드입니다. 
검색된 아래 리뷰 데이터(Context)를 기반으로 사용자가 찾는 게임이 무엇인지 파악하고 답하세요.
리뷰 본문 앞에 적힌 [게임명: ...] 정보를 활용하여 반드시 어떤 게임에 대한 설명인지 밝혀주세요.

Context: 
{context}

Question: 
{question}

Answer (in Korean):
반드시 아래 형식을 지켜주세요:
- 게임명: 
- 리뷰 요약: 
- 추천 이유: 
- 장점: 
"""
prompt = ChatPromptTemplate.from_template(template)

# 2. 실행 함수 수정 (메타데이터의 게임명을 컨텍스트에 포함)
def ask_game_recommendation(query):
    # 관련 리뷰 검색
    docs = retriever.invoke(query)
    
    # [중요] 리뷰 본문과 메타데이터의 게임명을 합쳐서 LLM에게 전달
    context_list = []
    for d in docs:
        game_name = d.metadata.get('game_name', '알 수 없음')
        context_list.append(f"[게임명: {game_name}] {d.page_content}")
    
    context = "\n".join(context_list)
    
    # LLM 답변 생성
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": query})
    
    return response.content

# 테스트 실행
question = "감동적인 스토리가 포함된 게임 추천 해줘."
result = ask_game_recommendation(question)
print(result)

- 게임명: Limbus Company  
- 리뷰 요약: 스토리가 겁나 재미있고 매력적이라는 리뷰가 다수 존재합니다.  
- 추천 이유: 감동적인 스토리를 찾고 있다면, Limbus Company는 매력적인 이야기 전개로 사용자에게 깊은 인상을 남길 수 있는 게임입니다.  
- 장점: 흥미로운 스토리라인, 독특한 캐릭터들, 그리고 몰입감 있는 게임플레이가 특징입니다.
